# Fine-tune Laya with decisionsmith (Kaggle or Colab)

**Kaggle:** Settings → Accelerator **GPU T4 x2**, Internet **on**. **Colab:** Runtime → Change runtime type → **T4 GPU**.

1. Put your labelled CSV (a `text` column + one column per field) or `h.export(...)` JSONL next to this notebook.
2. Edit the schema cell to match your fields.
3. Run all. Download `runs/v1.zip` and use it: `student="laya:./runs/v1"`.

Built on [Laya](https://github.com/NandhaKishorM/laya) (Apache-2.0) by Nandakishor M / Convai Innovations.

In [ ]:
!uv pip install --system -q "decisionsmith[laya]"  # Kaggle without uv: !pip install -q uv first
!nvidia-smi -L

In [ ]:
%%writefile schema.py
from typing import Annotated, Literal

from pydantic import BaseModel, Field

import decisionsmith as ds


class Ticket(BaseModel):
    team: Annotated[Literal["billing", "technical", "sales"],
                    ds.Options(billing="payments, refunds", technical="bugs, outages", sales="pricing")]
    wants_refund: bool = Field(description="Does the customer ask for their money back?")

In [ ]:
DATA = "train.csv"  # or train.jsonl from h.export(); on Kaggle: /kaggle/input/<dataset>/train.csv
import os

import torch

GPUS = torch.cuda.device_count()
print("GPUs:", GPUS, "| data found:", os.path.exists(DATA))

Train. With 2 GPUs (Kaggle T4 x2) this runs data-parallel; otherwise on one device. `--train full` fine-tunes the whole model; drop it to let decisionsmith pick (head-only below 1,000 rows).

In [ ]:
if GPUS > 1:
    !torchrun --standalone --nproc_per_node={GPUS} -m decisionsmith.cli finetune {DATA} --schema schema.py:Ticket --out runs/v1 --train full
else:
    !decisionsmith finetune {DATA} --schema schema.py:Ticket --out runs/v1 --train full

In [ ]:
import json

report = json.load(open("runs/v1/report.json"))
print("go:", report["go"])
for r in report["reasons"]:
    print(" -", r)
for row in report["rows"]:
    print(row)

In [ ]:
!rm -rf runs/v1/checkpoint_latest && cd runs && zip -qr v1.zip v1 && ls -lh v1.zip